In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

In [2]:
# Seed
random.seed(42)
np.random.seed(42)

In [3]:
# Global variables
CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
OUT_PATH = os.path.join(CURRENT_DIR, 'out')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

# Show info on the terminal about how the execution is going.
VERBOSE = True
# Even if there is a checkpoint, the model is retrained.
FORCE_TRAINING = True
# Name of this experiment that will appear in the result files.
SUBFIX_NAME = 'gradient'

In [4]:
# fetch dataset
support = fetch_ucirepo(id=880)

# data (as pandas dataframes)
df_data = support.data.features.copy()
y = support.data.targets

df_data['y'] = df_data['age']
df_data.drop(columns=['age'], inplace=True)

print(df_data.shape)
df_data.isna().sum()

(9105, 42)


sex            0
dzgroup        0
dzclass        0
num.co         0
edu         1634
income      2982
scoma          1
charges      172
totcst       888
totmcst     3475
avtisst       82
race          42
sps            1
aps            1
surv2m         1
surv6m         1
hday           0
diabetes       0
dementia       0
ca             0
prg2m       1649
prg6m       1633
dnr           30
dnrday        30
meanbp         1
wblc         212
hrt            1
resp           1
temp           1
pafi        2325
alb         3372
bili        2601
crea          67
sod            1
ph          2284
glucose     4500
bun         4352
urine       4862
adlp        5641
adls        2867
adlsc          0
y              0
dtype: int64

In [5]:
df_data.describe().T

,count,mean,std,min,25%,50%,75%,max
num.co,9105.0,1.868644,1.344409,0.000000,1.000000,2.000000,3.000000,9.000000e+00
edu,7471.0,11.747691,3.447743,0.000000,10.000000,12.000000,14.000000,3.100000e+01
scoma,9104.0,12.058546,24.636694,0.000000,0.000000,0.000000,9.000000,1.000000e+02
charges,8933.0,59995.787811,102648.778198,1169.000000,9740.000000,25024.000000,64598.000000,1.435423e+06
totcst,8217.0,30825.867768,45780.820986,0.000000,5929.566400,14452.734400,36087.937500,6.332120e+05
totmcst,5630.0,28828.877838,43604.261932,-102.719970,5177.404300,13223.500000,34223.601600,7.106820e+05
avtisst,9023.0,22.610928,13.233248,1.000000,12.000000,19.500000,31.666656,8.300000e+01
sps,9104.0,25.525872,9.899377,0.199982,19.000000,23.898438,30.199219,9.918750e+01
aps,9104.0,37.597979,19.903852,0.000000,23.000000,34.000000,49.000000,1.430000e+02
surv2m,9104.0,0.635870,0.248175,0.000000,0.507690,0.715942,0.825928,9.699707e-01


In [6]:
# delete variables with more than «na_threshold» missing values
na_threshold = 800
df_data = df_data.loc[:, df_data.isna().sum() <= na_threshold].copy()
print(df_data.shape)
df_data.isna().sum()

(9105, 27)


sex           0
dzgroup       0
dzclass       0
num.co        0
scoma         1
charges     172
avtisst      82
race         42
sps           1
aps           1
surv2m        1
surv6m        1
hday          0
diabetes      0
dementia      0
ca            0
dnr          30
dnrday       30
meanbp        1
wblc        212
hrt           1
resp          1
temp          1
crea         67
sod           1
adlsc         0
y             0
dtype: int64

In [7]:
# delete rows with missing values
df_data.dropna(inplace=True)
print(df_data.shape)
df_data.isna().sum()

(8579, 27)


sex         0
dzgroup     0
dzclass     0
num.co      0
scoma       0
charges     0
avtisst     0
race        0
sps         0
aps         0
surv2m      0
surv6m      0
hday        0
diabetes    0
dementia    0
ca          0
dnr         0
dnrday      0
meanbp      0
wblc        0
hrt         0
resp        0
temp        0
crea        0
sod         0
adlsc       0
y           0
dtype: int64

In [8]:
# delete not useful categorical variables
df_data.drop(columns=['dzclass', 'ca', 'dnr'], inplace=True)

In [9]:
df_data = pd.get_dummies(df_data, columns=['sex', 'dzgroup', 'race'], drop_first=True)
dummy_columns = df_data.columns[df_data.columns.str.startswith(('sex', 'dzgroup', 'race'))]
df_data[dummy_columns] = df_data[dummy_columns].astype(int)

In [10]:
df_data.shape

(8579, 33)

In [11]:
df_data.nunique()

num.co                    10
scoma                     11
charges                 8193
avtisst                  350
sps                      593
aps                      124
surv2m                   945
surv6m                   933
hday                      84
diabetes                   2
dementia                   2
dnrday                   172
meanbp                   164
wblc                     492
hrt                      184
resp                      66
temp                      98
crea                     129
sod                       60
adlsc                   1614
y                       7007
sex_male                   2
dzgroup_CHF                2
dzgroup_COPD               2
dzgroup_Cirrhosis          2
dzgroup_Colon Cancer       2
dzgroup_Coma               2
dzgroup_Lung Cancer        2
dzgroup_MOSF w/Malig       2
race_black                 2
race_hispanic              2
race_other                 2
race_white                 2
dtype: int64

In [12]:
# assert there is no more categorical variables
columnas_categoricas = df_data.select_dtypes(include=['object', 'category']).columns
columnas_categoricas

Index([], dtype='object')

In [14]:
df_data.to_parquet(
    os.path.join(
        DATA_PATH,
        'tabular',
        'real',
        'support',
        'clean.parquet'
    )
)